# Lab 5 — Stemming
*NLP · answers notebook*

In [1]:
import re
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer
from nltk.tokenize import RegexpTokenizer

## Exercise 1 — Morphological variation (by hand)

**1. Grouping.** All five words (*connect, connected, connecting, connection, connections*) express the same idea — the act/state of joining things — so a search engine should treat them as **one group**. They differ only by inflection (`-ed`, `-ing`, plural `-s`) and derivation (`-ion`) that change tense, number or part of speech, not the core meaning.

**2. Common stem.** `connect`

**3. The *argue* family** (argue, argued, arguing, argument, arguably). Hand-made guess: `argu`.
It is **not** as easy as for *connect*:
- *argue / argued / arguing* → clean stem `argu` (the final `-e` is dropped before `-ed`/`-ing`, so the stem isn't even a spelling of the base word).
- *argument* keeps its `e` (`argu` + `-ment`), so a purely suffix-stripping rule has to decide whether to remove `-ment` or not.
- *arguably* is a derivation through `-able` + `-ly` and also has a slightly different meaning ("it can be argued"), so it's debatable whether it belongs to the same group at all.

### Questions
1. **Similarities in the *connect* family:** they all share the identical prefix `connect`; the only differences are suffixes (`-ed`, `-ing`, `-ion`, `-s`) that carry grammatical information (tense, aspect, number, noun-vs-verb) rather than new lexical meaning.
2. **Why a search engine wants them related:** a user searching "connecting devices" also wants documents containing "connected device" or "connection". Treating them as one term increases **recall**, shrinks the index/vocabulary, and reduces sparsity of term-frequency statistics (counts of the variants are pooled).
3. **Was *argue* harder?** Yes. The spelling changes at the morpheme boundary (`e` deletion), a derivational suffix (`-ment`) and a borderline semantic case (*arguably*) make the result less clean. This suggests stemming is **heuristic, rule-based string manipulation, not true linguistic analysis**: it works well on regular patterns but can't be perfect, and the stem isn't guaranteed to be a real word or to group exactly the "right" set of words.

## Exercise 2 — Porter Stemmer

In [2]:
stemmer = PorterStemmer()

words = ["connect", "connected", "connecting", "connections"]

for word in words:
    print(word, "->", stemmer.stem(word))

connect -> connect
connected -> connect
connecting -> connect
connections -> connect


**2.1** The Porter output is `connect` for every word, which **matches** the hand-made stem from Exercise 1.

**2.2 The *argue* family:**

In [3]:
argue_family = ["argue", "argued", "arguing", "argument", "arguably"]
for word in argue_family:
    print(f"{word:10s} -> {stemmer.stem(word)}")

argue      -> argu
argued     -> argu
arguing    -> argu
argument   -> argument
arguably   -> arguabl


Porter gives `argu` for *argue / argued / arguing* (same as my guess), but **`argument`** stays unchanged (Porter only strips `-ment` when the remaining stem has measure > 1, and `argu` has measure 1) and **`arguably` → `arguabl`**. So Porter splits the family into **three different stems** (`argu`, `argument`, `arguabl`), whereas my hand grouping put all five together — a clear case of **under-stemming**.

**2.3 – 2.4 `stem_paragraph` and vocabulary reduction**

In [4]:
tokenizer = RegexpTokenizer(r"[A-Za-z]+(?:'[A-Za-z]+)?")   # words only, no punctuation

def stem_paragraph(text, stemmer):
    """Tokenize a paragraph and return the list of stems (lower-cased)."""
    tokens = tokenizer.tokenize(text.lower())
    return [stemmer.stem(tok) for tok in tokens]

paragraph = (
    "Natural language processing studies how computers process and understand human languages. "
    "Researchers generously share relational databases, national corpora and organizational resources. "
    "A classifier is trained on labeled examples, and training requires many labeled documents. "
    "Searching large collections of texts becomes easily manageable when related words are grouped together. "
    "Stemming groups words like connected, connecting and connection into a single stem."
)

tokens = tokenizer.tokenize(paragraph.lower())
stems  = stem_paragraph(paragraph, stemmer)

print("Number of tokens         :", len(tokens))
print("Vocabulary before (types):", len(set(tokens)))
print("Vocabulary after  (types):", len(set(stems)))
reduction = 100 * (len(set(tokens)) - len(set(stems))) / len(set(tokens))
print(f"Reduction                : {reduction:.1f} %")
print()
print(list(zip(tokens, stems))[:20])

Number of tokens         : 60
Vocabulary before (types): 54
Vocabulary after  (types): 46
Reduction                : 14.8 %

[('natural', 'natur'), ('language', 'languag'), ('processing', 'process'), ('studies', 'studi'), ('how', 'how'), ('computers', 'comput'), ('process', 'process'), ('and', 'and'), ('understand', 'understand'), ('human', 'human'), ('languages', 'languag'), ('researchers', 'research'), ('generously', 'gener'), ('share', 'share'), ('relational', 'relat'), ('databases', 'databas'), ('national', 'nation'), ('corpora', 'corpora'), ('and', 'and'), ('organizational', 'organiz')]


## Exercise 3 — Over-stemming and under-stemming

In [5]:
ex3 = ["studies", "studying", "relational", "organization", "easily", "university", "universe"]
ex3_out = {w: stemmer.stem(w) for w in ex3}
for w, s in ex3_out.items():
    print(f"{w:14s} -> {s}")

studies        -> studi
studying       -> studi
relational     -> relat
organization   -> organ
easily         -> easili
university     -> univers
universe       -> univers


In [6]:
# over-stemming: different words that collapse to the same stem
from collections import defaultdict
groups = defaultdict(list)
for w, s in ex3_out.items():
    groups[s].append(w)
print({s: ws for s, ws in groups.items() if len(ws) > 1})

{'studi': ['studies', 'studying'], 'univers': ['university', 'universe']}


**3.1** Outputs (Porter): `studies→studi`, `studying→studi`, `relational→relat`, `organization→organ`, `easily→easili`, `university→univers`, `universe→univers`.

**3.2 Over-stemming:** `university` and `universe` both become **`univers`**, although they mean very different things (an educational institution vs. the cosmos).

**3.3 Under-stemming:** within this list there is no pair that clearly *should* share a stem but doesn't (`studies`/`studying` do match). The closest case is `easily → easili`: its base word `easy` stems to `easi`, so *easy* and *easily* end up in different buckets. That is under-stemming, though it involves a word that isn't in the list.

**3.4 Why over-stemming hurts search:** a user searching for "university admission" would also retrieve documents about the "universe" (astronomy articles), lowering **precision** and returning irrelevant results; the engine can no longer distinguish the two concepts once they share an index term.

### Questions
1. **Are all stems real words?** No. `studi`, `easili`, `univers`, `relat`, `organ` (*organ* is a word, but not the sense we want) are not valid / not meaningful stems — e.g. **`studi`** and **`easili`** are not English words.
2. **Is a dictionary word needed?** No. A stem is only an **index key**: what matters is that related words map to the same key and unrelated words (ideally) to different ones. It is never shown to the user, so `studi` works as well as `study` as long as it's applied consistently to both the documents and the query.
3. **Risk of aggressive stemming:** conflating unrelated words (over-stemming), as with *university/universe*, which reduces **precision** — the more aggressively suffixes are cut, the higher the risk of merging words with different meanings.

## Exercise 4 — Comparing stemming algorithms

In [7]:
porter, snowball, lancaster = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer()

ex4 = ["studies", "studying", "connection", "easily", "relational", "generously", "national"]

import pandas as pd
table = pd.DataFrame(
    {"Word": ex4,
     "Porter":    [porter.stem(w)    for w in ex4],
     "Snowball":  [snowball.stem(w)  for w in ex4],
     "Lancaster": [lancaster.stem(w) for w in ex4]}
).set_index("Word")
table

,Porter,Snowball,Lancaster
Word,,,
studies,studi,studi,study
studying,studi,studi,study
connection,connect,connect,connect
easily,easili,easili,easy
relational,relat,relat,rel
generously,gener,generous,gen
national,nation,nation,nat


In [8]:
# Which outputs are valid English words? Judged manually (no dictionary data needed).
valid = {"study", "connect", "easy", "nation", "generous"}
counts = {col: int(table[col].isin(valid).sum()) for col in table.columns}
counts

{'Porter': 2, 'Snowball': 3, 'Lancaster': 4}

**Valid-word count** (outputs that are real English words, out of 7):

| | Porter | Snowball | Lancaster |
|---|---|---|---|
| valid (of 7) | **2** (`connect`, `nation`) | **3** (`connect`, `generous`, `nation`) | **4** (`study` ×2, `connect`, `easy`) |

(Porter: `studi, studi, connect, easili, relat, gener, nation`; Snowball: `studi, studi, connect, easili, relat, generous, nation`; Lancaster: `study, study, connect, easy, rel, gen, nat`.)
> Note: `rel`, `gen`, `nat` are word fragments/abbreviations, not valid words in this sense.

**Word with the most disagreement: `generously`** — Porter → `gener`, Snowball → `generous`, Lancaster → `gen`: three different stems of three different lengths. (`relational` is a close second: `relat` / `relat` / `rel`.)
- **Porter** strips `-ly` and then the `-ous` suffix, over-trimming to `gener` (which also collides with *general*, *generate*…).
- **Snowball** (Porter2) is a refined version of Porter that handles `-ously` more conservatively and removes only the `-ly`, leaving `generous`.
- **Lancaster** (Paice/Husk) is the most aggressive: it applies an iterative rule table that keeps chopping while the remaining stem is ≥ 2 characters, so it reduces the word all the way to `gen`.

### Question — why do the results differ, and which to trust?
Each algorithm uses a **different rule set and different stopping conditions** (Porter: 5 ordered suffix-stripping steps with "measure" conditions; Snowball: corrected/extended Porter rules; Lancaster: a single iterative, table-driven, very aggressive loop). They trade off **precision vs. recall** differently.

For a production search engine I would trust **Snowball (English)**: it is deterministic, fast, well-tested, fixes known Porter issues, and strikes a balance — it conflates real inflectional variants without the severe over-stemming of Lancaster (`gen`, `nat`, `rel` merge many unrelated words). In practice one would also consider a lemmatizer if precision matters more than speed.

### Challenge — all three stemmers on the full paragraph

In [9]:
results = {}
for name, st in [("Porter", porter), ("Snowball", snowball), ("Lancaster", lancaster)]:
    st_tokens = stem_paragraph(paragraph, st)
    results[name] = len(set(st_tokens))

base = len(set(tokens))
summary = pd.DataFrame({
    "vocab size": results,
    "reduction %": {k: round(100*(base-v)/base, 1) for k, v in results.items()},
})
print("Original vocabulary:", base)
summary.sort_values("vocab size")

Original vocabulary: 54


,vocab size,reduction %
Lancaster,45,16.7
Porter,46,14.8
Snowball,46,14.8


In [10]:
# smaller vocabulary = more aggressive; ties are listed together
order = sorted(results.items(), key=lambda kv: kv[1])
print("Most -> least aggressive:", " >= ".join(f"{n} ({v})" for n, v in order))

Most -> least aggressive: Lancaster (45) >= Porter (46) >= Snowball (46)


**Ranking:** the stemmer with the smallest vocabulary is the most aggressive, as printed above. This matches Exercise 4: Lancaster is the most aggressive, and Snowball is slightly more conservative than Porter on words like *generously*. If two stemmers tie on a short paragraph, that only means they merge the same groups of words there, not that they behave identically. Their stems for individual words can still differ.